In [ ]:
import torchvision.transforms as T
import numpy as np
import os
import torch
from common import *
from matplotlib import pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import PIL
import PIL.Image
import cv2
from kemsekov_torch.positional_emb import PositionalEncodingPermute
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.ndimage import affine_transform
import random
import skimage.measure
from scipy.ndimage import rotate
from scipy.spatial.transform import Rotation



In [ ]:

class GeneratedDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, labels_dir, info_dir, images_transform=None, projections_order=['a', 'b']):
        super().__init__()
        
        self.projections_order = projections_order
        
        # Считываем разметку
        labels = os.listdir(labels_dir)
        index_to_label = {i.split('.')[0]: os.path.join(labels_dir, i) for i in labels}
        
        # Считываем изображения
        images = os.listdir(images_dir)
        image_labels = [i.split("image")[1].split(".")[0] for i in images]
        
        data = {}
        for image, label in zip(images, image_labels):
            label_index = label[:-1]
            label_letter = label[-1]
            if label_index not in data:
                data[label_index] = {}
            data[label_index][label_letter] = os.path.join(images_dir, image)
        
        # Добавляем метки и проекции
        for i in index_to_label:
            row = data[i]
            row['label'] = index_to_label[i]
            projections = [row[p] for p in projections_order]
            row['projections'] = projections
        
        # Загружаем углы проекций из info файлов
        self.angles = {}
        info_files = os.listdir(info_dir)
        for info_file in info_files:
            if info_file.endswith(".info.0"):  # Фильтруем нужные файлы
                info_path = os.path.join(info_dir, info_file)
                with open(info_path, 'r') as f:
                    info_data = json.load(f)
                case_id = info_file.split('.')[0]  # ID кейса
                self.angles[case_id] = {
                    'theta': info_data.get("theta_array", []),
                    'phi': info_data.get("phi_array", [])
                }
        
        self.data = data
        self.keys = list(self.data.keys())
        self.images_transform = images_transform if images_transform else lambda x: T.ToTensor()(x)
    
    def __len__(self):
        return len(self.keys)
    
    def __getitem__(self, index):
        key = self.keys[index]
        row = self.data[key]
        label = np.load(row['label'])
        images = [PIL.Image.open(v) for v in row['projections']]
        images = torch.concat([self.images_transform(i) for i in images])
        
        # Получаем углы для данного случая
        angles = self.angles.get(key, {'theta': [0] * len(self.projections_order), 'phi': [0] * len(self.projections_order)})
        theta = torch.tensor(angles['theta'], dtype=torch.float32)
        phi = torch.tensor(angles['phi'], dtype=torch.float32)
        
        return images, torch.tensor(label), theta, phi

In [ ]:

# Сюда нужно запихнуть путь к сгенерированным через генератор данные
# labels_dir = "/home/alexus/Desktop/For_Generator/data_my/content/vessel_tree_generator/data/test/labels/test/"
# images_dir = "/home/alexus/Desktop/For_Generator/data_my/content/vessel_tree_generator/data/test/images/test/"
# info_dir = "/home/alexus/Desktop/For_Generator/data_my/content/vessel_tree_generator/data/test/info/"


labels_dir = "/home/alexus/Desktop/For_Generator/data0_1000/content/vessel_tree_generator/data/test/labels/test/"
images_dir = "/home/alexus/Desktop/For_Generator/data0_1000/content/vessel_tree_generator/data/test/images/test/"
info_dir = "/home/alexus/Desktop/For_Generator/data0_1000/content/vessel_tree_generator/data/test/info/"

def mean_channel(x):
    return x.mean(axis=0)[None,:]

def distance_transform(image_mask):
    image_mask=np.asarray(image_mask)[0]
    # Ensure the mask is binary
    binary_mask = (image_mask > np.mean(image_mask)).astype(np.uint8)
    # Apply the distance transform
    distance = cv2.distanceTransform(binary_mask, cv2.DIST_L2, 5)
    return torch.tensor(image_mask+distance[None,:])

images_transform=T.Compose([
    T.ToTensor(),
    T.Lambda(mean_channel),
    # T.Lambda(add_pos_encoding),
    # T.Lambda(lambda x: crop_to_content(x,0.5)),
    # T.CenterCrop((400,400)),
    
    T.Resize((128,128)),
    T.Lambda(distance_transform),
])
d2 = GeneratedDataset(images_dir,labels_dir,info_dir,images_transform=images_transform)

In [ ]:
sample_index = 0 #random.randint(0,len(d)-1)
sample = d2[sample_index]
plt.figure(figsize=(15,15))
plt.subplot(1,4,1)
plt.imshow(sample[0][0])
plt.axis('off')
plt.subplot(1,4,2)
plt.imshow(sample[0][1])
plt.axis('off')


#Тени#

In [ ]:
images, label, theta, phi = d2[sample_index]
proj1 = images[0].cpu().numpy()
proj2 = images[1].cpu().numpy()
phi1, phi2 =phi[0].item(), phi[1].item()
theta1, theta2 = theta[0].item(), theta[1].item()
# Normalize projections if needed (assuming values are in [0, 4.6] from your debug output)
if proj1.max() > 1.0:
    proj1 = proj1 / proj1.max()  # Normalize to [0, 1]
    proj2 = proj2 / proj2.max()

In [ ]:
print(f'phi1 = {phi1}, phi2 = {phi2}, theta1 = {theta1}, theta2 = {theta2}', )

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
import skimage.measure


def place_projection_in_3d_with_shadow(projection, phi, theta, volume_shape=(128, 128, 128), max_depth=128, threshold=0.3):
    D = volume_shape[0]
    
    # Бинаризуем изображение с сохранением интенсивностей
    binary_proj = torch.tensor((projection > threshold) * projection, dtype=torch.float32)

    h, w = binary_proj.shape
    
    # Создаём пустой 3D volume
    vol = torch.zeros(volume_shape, dtype=torch.float32)
    
    # Центрируем проекцию в XY-плоскости на Z=0
    x0 = (D - w) // 2
    y0 = (D - h) // 2
    vol[y0:y0+h, x0:x0+w, 0] = binary_proj
    
    # Получаем координаты ненулевых точек
    y_coords, x_coords = torch.where(vol[:, :, 0] > 0)
    intensities = vol[y0:y0+h, x0:x0+w, 0][y_coords - y0, x_coords - x0]

    # Переводим углы в радианы
    phi_rad = torch.tensor(np.radians(phi))
    theta_rad = torch.tensor(np.radians(theta))

    # Вычисляем направления теней
    dx = torch.cos(phi_rad) * torch.cos(theta_rad)
    dy = torch.sin(phi_rad) * torch.cos(theta_rad)
    dz = torch.sin(theta_rad)

    # Нормализация вектора направления
    norm = torch.sqrt(dx**2 + dy**2 + dz**2)
    dx, dy, dz = dx / norm, dy / norm, dz / norm

    # Ограничиваем глубину (max_depth вместо max_z)
    z_indices = torch.linspace(0, max_depth, max_depth).long()

    # Вычисляем координаты теней с ограниченной глубиной
    shadow_x = x_coords[:, None] + dx * z_indices[None, :]
    shadow_y = y_coords[:, None] + dy * z_indices[None, :]
    shadow_z = z_indices[None, :].repeat(len(x_coords), 1)

    # Округляем координаты до индексов и ограничиваем в пределах volume_shape
    shadow_x = shadow_x.round().long().clamp(0, D - 1)
    shadow_y = shadow_y.round().long().clamp(0, D - 1)
    shadow_z = shadow_z.clamp(0, D - 1)

    # Заполняем объём тенями с учетом интенсивностей
    for i in range(len(x_coords)):
        for j in range(max_depth):
            if shadow_z[i, j] < D:  # Убеждаемся, что не выходим за пределы
                vol[shadow_y[i, j], shadow_x[i, j], shadow_z[i, j]] = intensities[i]

    return vol

def build_3d_two_channels_with_shadow(proj1, phi1, theta1, proj2, phi2, theta2, volume_shape=(128, 128, 128), max_depth=30, threshold=0.3):
    # Создаём тени
    vol1 = place_projection_in_3d_with_shadow(proj1, phi1, theta1, volume_shape, max_depth, threshold)
    vol2 = place_projection_in_3d_with_shadow(proj2, phi2, theta2, volume_shape, max_depth, threshold)
    return torch.stack([vol1, vol2], dim=0)  # (2, 128, 128, 128)

# Генерируем 3D тензор с тенями
volume_2ch = build_3d_two_channels_with_shadow(proj1, phi1, theta1, proj2, phi2, theta2)


# Визуализация теней
ch1 = volume_2ch[0].cpu().numpy()
ch2 = volume_2ch[1].cpu().numpy()

# Применяем Marching Cubes для изоверхности
verts1, faces1, _, _ = skimage.measure.marching_cubes(ch1, level=0.1)
verts2, faces2, _, _ = skimage.measure.marching_cubes(ch2, level=0.1)

x1, y1, z1 = verts1[:, 0], verts1[:, 1], verts1[:, 2]
i1, j1, k1 = faces1[:, 0], faces1[:, 1], faces1[:, 2]

x2, y2, z2 = verts2[:, 0], verts2[:, 1], verts2[:, 2]
i2, j2, k2 = faces2[:, 0], faces2[:, 1], faces2[:, 2]

fig = go.Figure(data=[
    go.Mesh3d(x=x1, y=y1, z=z1, i=i1, j=j1, k=k1, opacity=0.5, color='steelblue', name='Shadow 1'),
    go.Mesh3d(x=x2, y=y2, z=z2, i=i2, j=j2, k=k2, opacity=0.5, color='green', name='Shadow 2')
])

fig.update_layout(
    title="3D Surface from Backprojection (Without Walls)",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=800,
    height=800,
)

fig.show()

In [ ]:

# Конвертируем тензор в numpy (если он в torch)
ch1 = volume_2ch[0].cpu().numpy()
ch2 = volume_2ch[1].cpu().numpy()

# Применяем алгоритм Marching Cubes для изоверхности
verts, faces, _, _ = skimage.measure.marching_cubes(ch1, level=0)
verts2, faces2, _, _ = skimage.measure.marching_cubes(ch2, level=0)

x, y, z = verts[:, 0], verts[:, 1], verts[:, 2]
i, j, k = faces[:, 0], faces[:, 1], faces[:, 2]

x2, y2, z2 = verts2[:, 0], verts2[:, 1], verts2[:, 2]
i2, j2, k2 = faces2[:, 0], faces2[:, 1], faces2[:, 2]

fig = go.Figure(data=[
    go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, opacity=1, color='steelblue'),
    go.Mesh3d(x=x2, y=y2, z=z2, i=i2, j=j2, k=k2, opacity=1, color='green')
])

fig.update_layout(
    title="3D Surface from Backprojection",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    width=800,
    height=800,
)

fig.show()


Работа с оригинальным тензором

In [ ]:
output_3d_model = np.load(labels_dir+'/0000.npy')
output_3d_model.shape

In [ ]:
import plotly.graph_objects as go

def create_cylinder(p0, p1, r0, r1, resolution=40):
    p0 = np.array(p0)
    p1 = np.array(p1)
    # Vector from p0 to p1
    v = p1 - p0
    v = v / np.linalg.norm(v)
    
    # Create two vectors orthogonal to v
    if (v == np.array([0, 0, 1])).all():
        ortho1 = np.array([1, 0, 0])
    else:
        ortho1 = np.cross(v, [0, 0, 1])
        ortho1 /= np.linalg.norm(ortho1)
    ortho2 = np.cross(v, ortho1)
    
    # Angles for the circle
    theta = np.linspace(0, 2 * np.pi, resolution, endpoint=False)
    
    # Create circle at p0
    circle0 = [p0 + r0 * (np.cos(t) * ortho1 + np.sin(t) * ortho2) for t in theta]
    # Create circle at p1
    circle1 = [p1 + r1 * (np.cos(t) * ortho1 + np.sin(t) * ortho2) for t in theta]
 
    # Combine vertices
    vertices = np.vstack((circle0, circle1))
    
    # Create faces
    faces = []
    for i in range(resolution):
        next_i = (i + 1) % resolution
        # Triangle 1
        faces.append((i, next_i, resolution + i))
        # Triangle 2
        faces.append((next_i, resolution + next_i, resolution + i))
    
    return vertices, faces

def create_tube(centerline, radius, resolution=40):
    all_vertices = []
    all_faces = []
    vertex_offset = 0

    for i in range(len(centerline) - 1):
        p0 = centerline[i]
        p1 = centerline[i + 1]
        r0 = radius[i]
        r1 = radius[i + 1]
        
        vertices, faces = create_cylinder(p0, p1, r0, r1, resolution)
        all_vertices.append(vertices)
        # Adjust face indices
        adjusted_faces = [(a + vertex_offset, b + vertex_offset, c + vertex_offset) for a, b, c in faces]
        all_faces.extend(adjusted_faces)
        vertex_offset += vertices.shape[0]
    
    all_vertices = np.vstack(all_vertices)
    return all_vertices, all_faces

def crop_to_content(image: torch.Tensor, alpha_threshold: float = 0.1) -> torch.Tensor:
    # Extract the alpha channel
    alpha = image[-1, :, :]

    # Create a binary mask where alpha is above the threshold
    mask = alpha > alpha_threshold

    # Find the bounding box of the content
    non_zero_indices = mask.nonzero(as_tuple=False)
    if non_zero_indices.numel() == 0:
        # If there's no content, return the original image
        return image

    y_min, x_min = non_zero_indices.min(dim=0)[0]
    y_max, x_max = non_zero_indices.max(dim=0)[0]

    # Crop the image to the bounding box
    cropped_image = image[:, y_min:y_max + 1, x_min:x_max + 1]

    return cropped_image

def render_sample(sample):
    meshes = []
    tensor_list = []

    for branch in sample:
        tensor = branch  # [x, y, z, r]
        tensor_list.append(tensor)

        centerline = branch[:, :3]
        radius = branch[:, 3]
        vertices, faces = create_tube(centerline, radius, resolution=40)

        x, y, z = vertices[:, 0], vertices[:, 1], vertices[:, 2]
        I = [face[0] for face in faces]
        J = [face[1] for face in faces]
        K = [face[2] for face in faces]

        mesh = go.Mesh3d(x=x, y=y, z=z, i=I, j=J, k=K, color='lightblue', opacity=0.6, name='Tube')
        scatter = go.Scatter3d(
            x=[point[0] for point in centerline],
            y=[point[1] for point in centerline],
            z=[point[2] for point in centerline],
            mode='lines+markers', line=dict(color='red', width=4),
            marker=dict(size=1, color='red'), name='Centerline'
        )
        meshes.append(mesh)
        meshes.append(scatter)

    combined_tensor = np.vstack(tensor_list)
    return combined_tensor, meshes


def normalize_tensor(sample):
    centerline_copy = sample.copy()
    slice = centerline_copy[...,:3]
    scale = slice.max()-slice.min()

    centerline_copy[...,:3]-=slice.min()
    centerline_copy[...,:3]/=slice.max()
    centerline_copy[...,-1]/=scale
    
    return centerline_copy 

output_3d_model = normalize_tensor(output_3d_model)
tensor, meshes = render_sample(output_3d_model)

# Визуализация (опционально)
layout = go.Layout(
    title='3D Tube Visualization',
    scene=dict(xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z'), aspectmode='data'),
    legend=dict(x=0.7, y=0.9), width=700, height=700
)
fig = go.Figure(data=meshes, layout=layout)
fig.show()

print("тензор:\n", tensor[:5])  # Первые 5 строк для примера


In [ ]:
output_3d_model 
output_3d_model.shape # 0 - батчи (ветки сосудов), 1 - кол.во точек, 2 - сами значения x,y,z,r

In [ ]:
# Для всех групп
T = torch.zeros((128, 128, 128), dtype=torch.float32)

for group in range(output_3d_model.shape[0]):
    x = (output_3d_model[group, :, 0] * 128).astype(int)
    y = (output_3d_model[group, :, 1] * 128).astype(int)
    z = (output_3d_model[group, :, 2] * 128).astype(int)
    R = torch.tensor(output_3d_model[group, :, 3], dtype=torch.float32)

    for i in range(len(x)):
        #radius = int(R)
        radius = int(R[i] * 128)  # Масштабируем радиус
        if radius < 1: radius = 1
        
        for shiftx in range(-radius, radius + 1):
            for shifty in range(-radius, radius + 1):
                for shiftz in range(-radius, radius + 1):
                    new_x = x[i] + shiftx
                    new_y = y[i] + shifty
                    new_z = z[i] + shiftz
                    if (0 <= new_x < 128 and 0 <= new_y < 128 and 0 <= new_z < 128):
                        if T[new_x, new_y, new_z] == 0:
                            T[new_x, new_y, new_z] = R[i]
                        else:
                            T[new_x, new_y, new_z] = (T[new_x, new_y, new_z] + R[i]) / 2
#T = T.unsqueeze(0)


In [ ]:

pio.templates.default = "plotly_dark"
plt.style.use('dark_background')
# Предположим, T уже создан (размер (128, 128, 128))
# Извлекаем координаты ненулевых элементов
#nonzero_indices = T.nonzero(as_tuple=False)  # Возвращает тензор формы (кол-во ненулевых, 3)
nonzero_indices = T.nonzero(as_tuple=False)  # Возвращает тензор формы (кол-во ненулевых, 3)
values = T[nonzero_indices[:, 0], nonzero_indices[:, 1], nonzero_indices[:, 2]]  # Значения в этих точках

# Преобразуем в списки для Plotly
x_coords = nonzero_indices[:, 0].tolist()
y_coords = nonzero_indices[:, 1].tolist()
z_coords = nonzero_indices[:, 2].tolist()
radii = values.tolist()  # Яркость или размер точек

# Создаём 3D Scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=x_coords,
    y=y_coords,
    z=z_coords,
    mode='markers',
    marker=dict(
        size=[r * 100 for r in radii],  # Увеличиваем радиус для видимости
        color=radii,                    # Цвет зависит от значения радиуса
        colorscale='Viridis',           # Цветовая шкала
        opacity=0.8,
        colorbar=dict(title="Radius")   # Добавляем шкалу
    )
)])


# Настраиваем внешний вид
fig.update_layout(
    title="3D Visualization of Tensor T",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode='cube'  # Кубическая форма для равных осей   
    ), width=700, height=700
)

# Показываем график
fig.show()

In [ ]:
output_3d_model = np.load(labels_dir+'/0000.npy')

In [ ]:
from ipywidgets import interact

volume_shape=(128, 128, 128) 
# Функция для создания тени от проекции
def create_shadow(projection, phi, theta, volume_shape=(128, 128, 128), threshold=0.1):
    D_x, D_y, D_z = volume_shape
    # Уменьшаем размер проекции до 64x64
    from scipy.ndimage import zoom
    projection = zoom(projection, (D_x / projection.shape[0], D_y / projection.shape[1]), order=1)
    binary_proj = torch.tensor((projection > threshold) * projection, dtype=torch.float32)
    h, w = binary_proj.shape
    vol = torch.zeros(volume_shape, dtype=torch.float32)
    x0, y0 = (D_x - w) // 2, (D_y - h) // 2
    vol[y0:y0+h, x0:x0+w, 0] = binary_proj
    y_coords, x_coords = torch.where(vol[:, :, 0] > 0)
    intensities = vol[y0:y0+h, x0:x0+w, 0][y_coords - y0, x_coords - x0]
    phi_rad, theta_rad = torch.tensor(np.radians(phi)), torch.tensor(np.radians(theta))
    dx = torch.cos(phi_rad) * torch.cos(theta_rad)
    dy = torch.sin(phi_rad) * torch.cos(theta_rad)
    dz = torch.sin(theta_rad)
    norm = torch.sqrt(dx**2 + dy**2 + dz**2)
    dx, dy, dz = dx / norm, dy / norm, dz / norm
    z_indices = torch.arange(D_z, dtype=torch.long)
    shadow_x = x_coords[:, None] + dx * z_indices[None, :]
    shadow_y = y_coords[:, None] + dy * z_indices[None, :]
    shadow_z = z_indices[None, :].repeat(len(x_coords), 1)
    shadow_x = shadow_x.round().long().clamp(0, D_x - 1)
    shadow_y = shadow_y.round().long().clamp(0, D_y - 1)
    shadow_z = shadow_z.clamp(0, D_z - 1)
    for i in range(len(x_coords)):
        for j in range(D_z):
            if shadow_z[i, j] < D_z:
                vol[shadow_y[i, j], shadow_x[i, j], shadow_z[i, j]] = intensities[i]
    return vol

# Функция для размещения сосуда с интерактивной высотой
def place_T_with_offset(T, z_offset, volume_shape=(128, 128, 128)):
    D_x, D_y, D_z = volume_shape
    T_shifted = torch.zeros(volume_shape, dtype=torch.float32)
    z_min = max(0, z_offset)
    z_max = min(D_z, T.shape[2] + z_offset)
    if z_max > z_min:
        T_shifted[:, :, z_min:z_max] = T[:, :, :z_max - z_min]
    return T_shifted

# Создаём тени
shadow1 = create_shadow(proj1, phi1, theta1, volume_shape, threshold=0.1)
shadow2 = create_shadow(proj2, phi2, theta2, volume_shape, threshold=0.1)

# Отладка теней и T
print("Shadow1 non-zero count:", torch.count_nonzero(shadow1).item())
print("Shadow2 non-zero count:", torch.count_nonzero(shadow2).item())
print("T non-zero count:", torch.count_nonzero(T).item())
print("Max value in shadow1:", shadow1.max().item())
print("Max value in shadow2:", shadow2.max().item())
print("Max value in T:", T.max().item())

# Интерактивная визуализация
def update_visualization(z_offset=0):
    T_placed = place_T_with_offset(T, z_offset, volume_shape)
    x = np.arange(volume_shape[0])
    y = np.arange(volume_shape[1])
    z = np.arange(volume_shape[2])
    fig = go.Figure()
    # Тень 1
    fig.add_trace(go.Isosurface(
        x=x, y=y, z=z,
        value=shadow1.numpy().flatten(),
        isomin=0.1, isomax=1.0,
        opacity=0.3, surface_count=10, colorscale='Blues', name='Shadow 1'
    ))
    # Тень 2
    fig.add_trace(go.Isosurface(
        x=x, y=y, z=z,
        value=shadow2.numpy().flatten(),
        isomin=0.1, isomax=1.0,
        opacity=0.3, surface_count=10, colorscale='Greens', name='Shadow 2'
    ))
    # Сосуд T (используем Scatter3d для ускорения)
    T_coords = torch.where(T_placed > 0)
    fig.add_trace(go.Scatter3d(
        x=T_coords[0].numpy(), y=T_coords[1].numpy(), z=T_coords[2].numpy(),
        mode='markers', marker=dict(size=2, color='red', opacity=0.6), name='Original T'
    ))
    fig.update_layout(
        title="3D Surface: Shadows and Original T with Interactive Z-Offset",
        scene=dict(
            xaxis=dict(range=[0, 128], title='X'),
            yaxis=dict(range=[0, 128], title='Y'),
            zaxis=dict(range=[0, 128], title='Z')
        ),
        width=800, height=800
    )
    fig.show()

# Запуск интерактивного управления
interact(update_visualization, z_offset=(0, 63, 1))

In [ ]:
from ipywidgets import interact
from scipy.ndimage import zoom


# Устанавливаем volume_shape
volume_shape = (128, 128, 128)

# Создание T из label с оптимизацией
T = torch.zeros(volume_shape, dtype=torch.float32)
output_3d_model = label.cpu().numpy()
output_3d_model = normalize_tensor(output_3d_model)

for group in range(output_3d_model.shape[0]):
    # Масштабируем координаты на диапазон [0, 127], сдвигаем к центру
    x = (output_3d_model[group, :, 0] * 100 + 10).astype(int).clip(0, 127)
    y = (output_3d_model[group, :, 1] * 100 + 10).astype(int).clip(0, 127)
    z = (output_3d_model[group, :, 2] * 100 + 10).astype(int).clip(0, 127)
    R = torch.tensor(output_3d_model[group, :, 3], dtype=torch.float32) * 10  # Увеличиваем интенсивность
    print(f"Group {group} R min, max:", R.min().item(), R.max().item())
    
    for i in range(len(x)):
        radius = int((R[i] * 20).clamp(1, 10))  # Уменьшаем радиус для ускорения
        x_grid, y_grid, z_grid = torch.meshgrid(
            torch.arange(-radius, radius + 1),
            torch.arange(-radius, radius + 1),
            torch.arange(-radius, radius + 1)
        )
        mask = (x_grid ** 2 + y_grid ** 2 + z_grid ** 2 <= radius ** 2)
        x_shifted = x[i] + x_grid[mask]
        y_shifted = y[i] + y_grid[mask]
        z_shifted = z[i] + z_grid[mask]
        valid = (x_shifted >= 0) & (x_shifted < 128) & (y_shifted >= 0) & (y_shifted < 128) & (z_shifted >= 0) & (z_shifted < 128)
        x_valid, y_valid, z_valid = x_shifted[valid], y_shifted[valid], z_shifted[valid]
        for x_v, y_v, z_v in zip(x_valid, y_valid, z_valid):
            if T[x_v, y_v, z_v] == 0:
                T[x_v, y_v, z_v] = R[i]
            else:
                T[x_v, y_v, z_v] = (T[x_v, y_v, z_v] + R[i]) / 2

In [ ]:
# Функция для создания тени от проекции
def create_shadow(projection, phi, theta, volume_shape=(128, 128, 128), threshold=0.1):
    D_x, D_y, D_z = volume_shape 
    projection = zoom(projection, (D_x / projection.shape[0], D_y / projection.shape[1]), order=1)
    binary_proj = torch.tensor((projection > threshold) * projection, dtype=torch.float32)
    h, w = binary_proj.shape
    vol = torch.zeros(volume_shape, dtype=torch.float32)
    x0, y0 = (D_x - w) // 2, (D_y - h) // 2
    vol[y0:y0+h, x0:x0+w, 0] = binary_proj
    y_coords, x_coords = torch.where(vol[:, :, 0] > 0)
    intensities = vol[y0:y0+h, x0:x0+w, 0][y_coords - y0, x_coords - x0]
    phi_rad, theta_rad = torch.tensor(np.radians(phi)), torch.tensor(np.radians(theta))
    dx = torch.cos(phi_rad) * torch.cos(theta_rad)
    dy = torch.sin(phi_rad) * torch.cos(theta_rad)
    dz = torch.sin(theta_rad)
    norm = torch.sqrt(dx**2 + dy**2 + dz**2)
    dx, dy, dz = dx / norm, dy / norm, dz / norm
    z_indices = torch.arange(D_z, dtype=torch.long)
    shadow_x = x_coords[:, None] + dx * z_indices[None, :]
    shadow_y = y_coords[:, None] + dy * z_indices[None, :]
    shadow_z = z_indices[None, :].repeat(len(x_coords), 1)
    shadow_x = shadow_x.round().long().clamp(0, D_x - 1)
    shadow_y = shadow_y.round().long().clamp(0, D_y - 1)
    shadow_z = shadow_z.clamp(0, D_z - 1)
    for i in range(len(x_coords)):
        for j in range(D_z):
            if shadow_z[i, j] < D_z:
                vol[shadow_y[i, j], shadow_x[i, j], shadow_z[i, j]] = intensities[i]
    return vol

# Функция для размещения сосуда с интерактивной высотой
def place_T_with_offset(T, z_offset, volume_shape=(128, 128, 128)):
    D_x, D_y, D_z = volume_shape
    T_shifted = torch.zeros(volume_shape, dtype=torch.float32)
    z_min = max(0, z_offset)
    z_max = min(D_z, T.shape[2] + z_offset)
    if z_max > z_min:
        T_shifted[:, :, z_min:z_max] = T[:, :, :z_max - z_min]
    return T_shifted

# Создаём тени
shadow1 = create_shadow(proj1, phi1, theta1, volume_shape, threshold=0.1)
shadow2 = create_shadow(proj2, phi2, theta2, volume_shape, threshold=0.1)

# Отладка теней и T
print("Shadow1 non-zero count:", torch.count_nonzero(shadow1).item())
print("Shadow2 non-zero count:", torch.count_nonzero(shadow2).item())
print("T non-zero count:", torch.count_nonzero(T).item())
print("Max value in shadow1:", shadow1.max().item())
print("Max value in shadow2:", shadow2.max().item())
print("Max value in T:", T.max().item())

# Интерактивная визуализация с поворотом
def update_visualization(z_offset=0, phi=phi2, theta=theta2):
    # Смещаем и поворачиваем T
    T_shifted = place_T_with_offset(T, z_offset, volume_shape)
    T_rotated = rotate_tensor(T_shifted, phi, theta, volume_shape)

    x = np.arange(volume_shape[0])
    y = np.arange(volume_shape[1])
    z = np.arange(volume_shape[2])

    fig = go.Figure()
    
    # Тень 1 (только точки с интенсивностью > 0.2)
    shadow1_coords = torch.where(shadow1 > 0.2)
    fig.add_trace(go.Scatter3d(
        x=shadow1_coords[0].numpy(), y=shadow1_coords[1].numpy(), z=shadow1_coords[2].numpy(),
        mode='markers', marker=dict(size=2, color='blue', opacity=0.9), name='Shadow 1'
    ))
    
    # Тень 2 (только точки с интенсивностью > 0.2)
    shadow2_coords = torch.where(shadow2 > 0.2)
    fig.add_trace(go.Scatter3d(
        x=shadow2_coords[0].numpy(), y=shadow2_coords[1].numpy(), z=shadow2_coords[2].numpy(),
        mode='markers', marker=dict(size=2, color='green', opacity=0.9), name='Shadow 2'
    ))
    
    # Сосуд T после поворота (только точки с интенсивностью > 0.01)
    T_coords = torch.where(T_rotated > 0.01)
    fig.add_trace(go.Scatter3d(
        x=T_coords[0].numpy(), y=T_coords[1].numpy(), z=T_coords[2].numpy(),
        mode='markers', marker=dict(size=3, color='red', opacity=0.5), name='Rotated T'
    ))
    
    fig.update_layout(
        title="3D Points: Shadows and Rotated T with Interactive Z-Offset and Rotation",
        scene=dict(
            xaxis=dict(range=[0, 128], title='X'),
            yaxis=dict(range=[0, 128], title='Y'),
            zaxis=dict(range=[0, 128], title='Z')
        ),
        width=1200, height=800
    )
    fig.show()

# Запуск интерактивного управления
interact(update_visualization, z_offset=(-63, 63, 1), phi=(-140, 120, 5), theta=(-140, 180, 5))

In [ ]:


import torch
import numpy as np
from sklearn.decomposition import PCA

# Извлечение координат ненулевых вокселей T
T_coords = torch.where(T > 0.01)
T_points = torch.stack(T_coords, dim=-1).float().numpy()  # (N, 3)

# Центрируем точки
T_points_centered = T_points - T_points.mean(axis=0)

# Применяем PCA для определения главной оси
pca = PCA(n_components=3)
pca.fit(T_points_centered)
principal_axis = pca.components_[0]  # Главная ось (вектор)

print("Главная ось T:", principal_axis)

In [ ]:
def compare_3d_structures(T1, T2, threshold=0.01):
    coords1 = torch.where(T1 > threshold)
    coords2 = torch.where(T2 > threshold)
    points1 = torch.stack(coords1, dim=-1).float()
    points2 = torch.stack(coords2, dim=-1).float()
    min_points = min(points1.shape[0], points2.shape[0])
    points1 = points1[:min_points]
    points2 = points2[:min_points]
    return torch.sum((points1 - points2) ** 2)

best_theta = 0
min_error = float('inf')
best_phi = -13
for theta in range(-180, 181, 5):
    rotated_T = rotate_tensor(T, best_phi, theta, volume_shape)
    error = compare_3d_structures(rotated_T, shadow1, threshold=0.2)
    if error < min_error:
        min_error = error
        best_theta = theta

print(f"Лучший theta (по 3D): {best_theta} (при phi={best_phi})")

In [ ]:
import torch

def shift_tensor_xy(tensor, shift_x, shift_y, volume_shape=(128, 128, 128)):
    """Смещает тензор только в плоскости XY."""
    D_x, D_y, D_z = volume_shape
    shifted_tensor = torch.zeros_like(tensor)
    
    # Извлекаем координаты ненулевых вокселей
    coords = torch.where(tensor > 0.01)
    points = torch.stack(coords, dim=-1).float()
    intensities = tensor[coords]
    
    # Применяем смещение только по X и Y
    shifted_points = points + torch.tensor([shift_x, shift_y, 0], dtype=torch.float)
    
    # Округляем и ограничиваем координаты
    shifted_points = shifted_points.round().long()
    valid = (shifted_points[:, 0] >= 0) & (shifted_points[:, 0] < D_x) & \
            (shifted_points[:, 1] >= 0) & (shifted_points[:, 1] < D_y) & \
            (shifted_points[:, 2] >= 0) & (shifted_points[:, 2] < D_z)
    
    shifted_points = shifted_points[valid]
    intensities = intensities[valid]
    
    # Заполняем новый тензор
    for i in range(len(shifted_points)):
        x, y, z = shifted_points[i]
        if shifted_tensor[x, y, z] == 0:
            shifted_tensor[x, y, z] = intensities[i]
        else:
            shifted_tensor[x, y, z] = (shifted_tensor[x, y, z] + intensities[i]) / 2
    
    return shifted_tensor


def center_tensor(tensor, volume_shape=(128, 128, 128)):
    center = torch.tensor(volume_shape, dtype=torch.float) / 2
    coords = torch.where(tensor > 0.01)
    points = torch.stack(coords, dim=-1).float()
    intensities = tensor[coords]
    current_center = points.mean(dim=0)
    shift = center - current_center
    return shift_tensor(tensor, shift[0].item(), shift[1].item(), shift[2].item(), volume_shape)

# Центрируем T перед обработкой
T = center_tensor(T, volume_shape)

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact

# Функция поворота
def rotate_tensor(tensor, phi, theta, volume_shape):
    D_x, D_y, D_z = volume_shape
    rotated_tensor = torch.zeros_like(tensor)

    phi_rad = np.radians(phi)
    theta_rad = np.radians(theta)

    Rz = torch.tensor([
        [np.cos(phi_rad), -np.sin(phi_rad), 0],
        [np.sin(phi_rad), np.cos(phi_rad), 0],
        [0, 0, 1]
    ], dtype=torch.float32)
    Ry = torch.tensor([
        [np.cos(theta_rad), 0, np.sin(theta_rad)],
        [0, 1, 0],
        [-np.sin(theta_rad), 0, np.cos(theta_rad)]
    ], dtype=torch.float32)
    R = torch.matmul(Ry, Rz)

    coords = torch.where(tensor > 0.01)
    points = torch.stack(coords, dim=-1).float() - torch.tensor([D_x/2, D_y/2, D_z/2], dtype=torch.float32)
    intensities = tensor[coords]

    rotated_points = torch.matmul(points, R.T)
    rotated_points = rotated_points + torch.tensor([D_x/2, D_y/2, D_z/2], dtype=torch.float32)
    rotated_points = rotated_points.round().long()

    valid = (rotated_points[:, 0] >= 0) & (rotated_points[:, 0] < D_x) & \
            (rotated_points[:, 1] >= 0) & (rotated_points[:, 1] < D_y) & \
            (rotated_points[:, 2] >= 0) & (rotated_points[:, 2] < D_z)

    rotated_points = rotated_points[valid]
    intensities = intensities[valid]

    for i in range(len(rotated_points)):
        x, y, z = rotated_points[i]
        if rotated_tensor[x, y, z] == 0:
            rotated_tensor[x, y, z] = intensities[i]

    return rotated_tensor

# Функция проекции
def project_3d_model(tensor, phi, theta, volume_shape=(128, 128, 128)):
    tensor_rotated = rotate_tensor(tensor, -phi, -theta, volume_shape)
    return torch.max(tensor_rotated, dim=2)[0]

# Инициализация
volume_shape = (128, 128, 128)

# Применяем углы ангиографа к теням
shadow1_rotated = rotate_tensor(shadow1, 100.0, -100.0, volume_shape)  # Угол для Shadow 1
shadow2_rotated = rotate_tensor(shadow2, 5.0, -75.0, volume_shape)    # Угол для Shadow 2

shadow1_coords = torch.where(shadow1_rotated > 0.2)
shadow2_coords = torch.where(shadow2_rotated > 0.2)

# Интерактивная визуализация
def update_3d_model(phi_T=15.0, theta_T=180.0, shift_x=-5, shift_y=-6, z_offset=-25):
    # Применяем поворот и сдвиг к T
    T_rotated = rotate_tensor(T, phi_T, theta_T, volume_shape)
    T_shifted = torch.roll(T_rotated, shifts=(shift_x, shift_y, z_offset), dims=(0, 1, 2))

    # Проекции для проверки
    proj_T1 = project_3d_model(T_shifted, 100.0, -100.0, volume_shape)
    proj_T2 = project_3d_model(T_shifted, 5.0, -75.0, volume_shape)

    # 3D-визуализация
    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=shadow1_coords[0].numpy(), y=shadow1_coords[1].numpy(), z=shadow1_coords[2].numpy(),
        mode='markers', marker=dict(size=2, color='blue', opacity=0.5), name='Shadow 1'
    ))
    fig.add_trace(go.Scatter3d(
        x=shadow2_coords[0].numpy(), y=shadow2_coords[1].numpy(), z=shadow2_coords[2].numpy(),
        mode='markers', marker=dict(size=2, color='green', opacity=0.5), name='Shadow 2'
    ))
    T_coords = torch.where(T_shifted > 0.01)
    fig.add_trace(go.Scatter3d(
        x=T_coords[0].numpy(), y=T_coords[1].numpy(), z=T_coords[2].numpy(),
        mode='markers', marker=dict(size=3, color='red', opacity=0.2), name='Rotated T'
    ))

    fig.update_layout(
        title="3D Reconstruction: Vessel and Shadows with Interactive Adjustment",
        scene=dict(
            xaxis=dict(range=[0, volume_shape[0]], title='X'),
            yaxis=dict(range=[0, volume_shape[1]], title='Y'),
            zaxis=dict(range=[0, volume_shape[2]], title='Z')
        ),
        width=800, height=800
    )
    fig.show()

    # Показываем 2D-проекции для контроля
    fig2, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(proj_T1.numpy(), cmap='viridis')
    axs[0].set_title(f'Proj_T1 (phi=100, theta=-100)')
    axs[1].imshow(proj_T2.numpy(), cmap='viridis')
    axs[1].set_title(f'Proj_T2 (phi=5, theta=-75)')
    plt.show()

# Запуск интерактивной визуализации
interact(update_3d_model,
         phi_T=(-180.0, 180.0, 5.0),
         theta_T=(-180.0, 180.0, 5.0),
         shift_x=(-50, 50, 1),
         shift_y=(-50, 50, 1),
         z_offset=(-50, 50, 1))

In [ ]:
# Финальная ориентация
phi_T_final = 45.0
theta_T_final = -30.0
shift_x_final = 10
shift_y_final = -5
z_offset_final = 20

T_rotated_final = rotate_tensor(T, phi_T_final, theta_T_final, volume_shape)
T_shifted_final = torch.roll(T_rotated_final, shifts=(shift_x_final, shift_y_final, z_offset_final), dims=(0, 1, 2))

# Проверка проекций
proj_T1_final = project_3d_model(T_shifted_final, 100.0, 6.0, volume_shape)
proj_T2_final = project_3d_model(T_shifted_final, 5.0, -75.0, volume_shape)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(proj_T1_final.numpy(), cmap='viridis')
axs[0].set_title('Final Proj_T1')
axs[1].imshow(proj_T2_final.numpy(), cmap='viridis')
axs[1].set_title('Final Proj_T2')
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

# Предполагаемые функции (уже определены ранее)
def rotate_tensor(tensor, phi, theta, volume_shape):
    D_x, D_y, D_z = volume_shape
    rotated_tensor = torch.zeros_like(tensor)

    phi_rad = np.radians(phi)
    theta_rad = np.radians(theta)

    Rz = torch.tensor([
        [np.cos(phi_rad), -np.sin(phi_rad), 0],
        [np.sin(phi_rad), np.cos(phi_rad), 0],
        [0, 0, 1]
    ], dtype=torch.float32)
    Ry = torch.tensor([
        [np.cos(theta_rad), 0, np.sin(theta_rad)],
        [0, 1, 0],
        [-np.sin(theta_rad), 0, np.cos(theta_rad)]
    ], dtype=torch.float32)
    R = torch.matmul(Ry, Rz)

    coords = torch.where(tensor > 0.01)
    points = torch.stack(coords, dim=-1).float() - torch.tensor([D_x/2, D_y/2, D_z/2], dtype=torch.float32)
    intensities = tensor[coords]

    rotated_points = torch.matmul(points, R.T)
    rotated_points = rotated_points + torch.tensor([D_x/2, D_y/2, D_z/2], dtype=torch.float32)
    rotated_points = rotated_points.round().long()

    valid = (rotated_points[:, 0] >= 0) & (rotated_points[:, 0] < D_x) & \
            (rotated_points[:, 1] >= 0) & (rotated_points[:, 1] < D_y) & \
            (rotated_points[:, 2] >= 0) & (rotated_points[:, 2] < D_z)

    rotated_points = rotated_points[valid]
    intensities = intensities[valid]

    for i in range(len(rotated_points)):
        x, y, z = rotated_points[i]
        if rotated_tensor[x, y, z] == 0:
            rotated_tensor[x, y, z] = intensities[i]

    return rotated_tensor

def project_3d_model(tensor, volume_shape=(128, 128, 128)):
    return torch.max(tensor, dim=2)[0]

# Эталонные проекции
proj1 = torch.max(shadow1, dim=2)[0]
proj2 = torch.max(shadow2, dim=2)[0]

# Интерактивная визуализация
volume_shape = (128, 128, 128)

def update_projections(phi1=100.0, theta1=130.0, phi2=99.309, theta2=16.342):
    T_rotated1 = rotate_tensor(T, phi1, theta1, volume_shape)
    T_rotated2 = rotate_tensor(T, phi2, theta2, volume_shape)

    proj_T1 = project_3d_model(T_rotated1, volume_shape)
    proj_T2 = project_3d_model(T_rotated2, volume_shape)

    fig, axs = plt.subplots(2, 2, figsize=(10, 10))
    axs[0, 0].imshow(proj1.numpy(), cmap='viridis')
    axs[0, 0].set_title('Proj1 (Shadow 1)')
    axs[0, 1].imshow(proj2.numpy(), cmap='viridis')
    axs[0, 1].set_title('Proj2 (Shadow 2)')
    axs[1, 0].imshow(proj_T1.numpy(), cmap='viridis')
    axs[1, 0].set_title(f'Proj_T1 (phi1={phi1:.2f}, theta1={theta1:.2f})')
    axs[1, 1].imshow(proj_T2.numpy(), cmap='viridis')
    axs[1, 1].set_title(f'Proj_T2 (phi2={phi2:.2f}, theta2={theta2:.2f})')
    plt.show()

# Интерактивные ползунки
interact(update_projections,
         phi1=(-180.0, 180.0, 5.0),
         theta1=(-180.0, 180.0, 5.0),
         phi2=(-180.0, 180.0, 5.0),
         theta2=(-180.0, 180.0, 5.0))

In [ ]:
import torch
import numpy as np

def align_vector_to_z(vector):
    """Возвращает матрицу вращения, которая переводит vector в [0, 0, 1]."""
    vector = vector / torch.norm(vector)  # Нормализуем
    target = torch.tensor([0, 0, 1], dtype=torch.float)

    # Если векторы уже совпадают, возвращаем единичную матрицу
    if torch.allclose(vector, target):
        return torch.eye(3)

    # Вычисляем ось вращения (векторное произведение)
    axis = torch.cross(vector, target)
    axis = axis / torch.norm(axis)

    # Угол вращения (через скалярное произведение)
    cos_theta = torch.dot(vector, target)
    theta = torch.acos(cos_theta)

    # Матрица вращения (формула Родригеша)
    K = torch.tensor([
        [0, -axis[2], axis[1]],
        [axis[2], 0, -axis[0]],
        [-axis[1], axis[0], 0]
    ])
    R = torch.eye(3) + torch.sin(theta) * K + (1 - torch.cos(theta)) * torch.matmul(K, K)
    return R

# Главная ось T
main_axis = torch.tensor([0.16674638, 0.02588737, 0.9856599], dtype=torch.float)
R_align = align_vector_to_z(main_axis)

# Поворачиваем T
def rotate_tensor_with_matrix(tensor, R, volume_shape):
    D_x, D_y, D_z = volume_shape
    rotated_tensor = torch.zeros_like(tensor)

    # Центрируем координаты
    coords = torch.where(tensor > 0.01)
    points = torch.stack(coords, dim=-1).float() - torch.tensor([D_x/2, D_y/2, D_z/2]).float()
    intensities = tensor[coords]

    # Применяем вращение
    rotated_points = torch.matmul(points, R.T)

    # Возвращаем в исходную систему координат
    rotated_points = rotated_points + torch.tensor([D_x/2, D_y/2, D_z/2]).float()
    rotated_points = rotated_points.round().long()

    # Ограничиваем координаты
    valid = (rotated_points[:, 0] >= 0) & (rotated_points[:, 0] < D_x) & \
            (rotated_points[:, 1] >= 0) & (rotated_points[:, 1] < D_y) & \
            (rotated_points[:, 2] >= 0) & (rotated_points[:, 2] < D_z)

    rotated_points = rotated_points[valid]
    intensities = intensities[valid]

    for i in range(len(rotated_points)):
        x, y, z = rotated_points[i]
        if rotated_tensor[x, y, z] == 0:
            rotated_tensor[x, y, z] = intensities[i]

    return rotated_tensor

# Выравниваем T
T_aligned = rotate_tensor_with_matrix(T, R_align, volume_shape)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from ipywidgets import interact, fixed

# Функция поворота
def rotate_tensor(tensor, phi, theta, volume_shape):
    D_x, D_y, D_z = volume_shape
    rotated_tensor = torch.zeros_like(tensor)

    # Преобразуем phi и theta в тензоры
    if isinstance(phi, (int, float)):
        phi = torch.tensor(float(phi))
    if isinstance(theta, (int, float)):
        theta = torch.tensor(float(theta))

    phi_rad = torch.deg2rad(phi)  # Вращение вокруг Z
    theta_rad = torch.deg2rad(theta)  # Наклон относительно Z

    # Матрицы вращения (сначала наклон theta, затем вращение phi)
    Ry = torch.stack([
        torch.stack([torch.cos(theta_rad), torch.zeros_like(theta_rad), torch.sin(theta_rad)]),
        torch.stack([torch.zeros_like(theta_rad), torch.ones_like(theta_rad), torch.zeros_like(theta_rad)]),
        torch.stack([-torch.sin(theta_rad), torch.zeros_like(theta_rad), torch.cos(theta_rad)])
    ])
    Rz = torch.stack([
        torch.stack([torch.cos(phi_rad), -torch.sin(phi_rad), torch.zeros_like(phi_rad)]),
        torch.stack([torch.sin(phi_rad), torch.cos(phi_rad), torch.zeros_like(phi_rad)]),
        torch.stack([torch.zeros_like(phi_rad), torch.zeros_like(phi_rad), torch.ones_like(phi_rad)])
    ])
    R = torch.matmul(Rz, Ry)  # Порядок: сначала наклон, затем вращение

    # Координаты ненулевых точек
    coords = torch.where(tensor > 0.01)
    points = torch.stack(coords, dim=-1).float() - torch.tensor([D_x/2, D_y/2, D_z/2], dtype=torch.float32)
    intensities = tensor[coords]

    # Поворот точек
    rotated_points = torch.matmul(points, R.T)
    rotated_points = rotated_points + torch.tensor([D_x/2, D_y/2, D_z/2], dtype=torch.float32)
    rotated_points = rotated_points.round().long()

    # Проверка границ
    valid = (rotated_points[:, 0] >= 0) & (rotated_points[:, 0] < D_x) & \
            (rotated_points[:, 1] >= 0) & (rotated_points[:, 1] < D_y) & \
            (rotated_points[:, 2] >= 0) & (rotated_points[:, 2] < D_z)

    rotated_points = rotated_points[valid]
    intensities = intensities[valid]

    # Заполняем новый тензор
    for i in range(len(rotated_points)):
        x, y, z = rotated_points[i]
        if rotated_tensor[x, y, z] == 0:
            rotated_tensor[x, y, z] = intensities[i]

    return rotated_tensor

# Функция проекции на плоскость XY
def project_xy(tensor, phi, theta, volume_shape):
    tensor_rotated = rotate_tensor(tensor, -phi, -theta, volume_shape)  # Инверсия углов для соответствия ангиографу
    return torch.sum(tensor_rotated, dim=2)  # Проекция на XY (суммируем по Z)

# Эталонные проекции
volume_shape = T.shape
proj1 = torch.sum(shadow1, dim=2).detach()  # 2D-проекции теней
proj2 = torch.sum(shadow2, dim=2).detach()

# Углы ангиографа
phi1_orig = -100.0  # Вращение вокруг Z
theta1_orig = -130.0  # Наклон относительно Z
phi2_orig = -99.30917358398438
theta2_orig = -16.34215545654297

# Интерактивная визуализация с подсказками
def update_visualization(phi_T=30.0, theta_T=-140.0, shift_x=19, shift_y=-30, z_offset=5, invert_phi=False, invert_theta=False):
    # Корректировка углов с учетом инверсии
    phi_T_adjusted = -phi_T if invert_phi else phi_T
    theta_T_adjusted = -theta_T if invert_theta else theta_T

    # Поворачиваем и сдвигаем сосуд
    T_rotated = rotate_tensor(T, phi_T_adjusted, theta_T_adjusted, volume_shape)
    T_shifted = torch.roll(T_rotated, shifts=(int(shift_x), int(shift_y), int(z_offset)), dims=(0, 1, 2))

    # Проекции сосуда под ангиографическими углами
    proj_T1 = project_xy(T_shifted, phi1_orig, theta1_orig, volume_shape)
    proj_T2 = project_xy(T_shifted, phi2_orig, theta2_orig, volume_shape)

    # Вычисляем функцию потерь
    loss1 = torch.mean((proj_T1 - proj1) ** 2).item()
    loss2 = torch.mean((proj_T2 - proj2) ** 2).item()
    total_loss = loss1 + loss2

    # 3D-визуализация
    fig_3d = go.Figure()

    # Визуализация сосуда
    T_coords = torch.where(T_shifted > 0.01)
    fig_3d.add_trace(go.Scatter3d(
        x=T_coords[0].detach().numpy(), y=T_coords[1].detach().numpy(), z=T_coords[2].detach().numpy(),
        mode='markers', marker=dict(size=3, color='red', opacity=0.2), name='Rotated T'
    ))

    # Визуализация shadow1
    shadow1_coords = torch.where(shadow1 > 0.01)
    fig_3d.add_trace(go.Scatter3d(
        x=shadow1_coords[0].detach().numpy(), y=shadow1_coords[1].detach().numpy(), z=shadow1_coords[2].detach().numpy(),
        mode='markers', marker=dict(size=2, color='blue', opacity=0.1), name='Shadow 1'
    ))

    # Визуализация shadow2
    shadow2_coords = torch.where(shadow2 > 0.01)
    fig_3d.add_trace(go.Scatter3d(
        x=shadow2_coords[0].detach().numpy(), y=shadow2_coords[1].detach().numpy(), z=shadow2_coords[2].detach().numpy(),
        mode='markers', marker=dict(size=2, color='green', opacity=0.1), name='Shadow 2'
    ))

    # Аннотация осей
    fig_3d.add_trace(go.Scatter3d(
        x=[0, volume_shape[0]//2], y=[0, 0], z=[0, 0],
        mode='lines', line=dict(color='black', width=2), name='X (Left-Right)'
    ))
    fig_3d.add_trace(go.Scatter3d(
        x=[0, 0], y=[0, volume_shape[1]//2], z=[0, 0],
        mode='lines', line=dict(color='black', width=2), name='Y (Head-Feet)'
    ))
    fig_3d.add_trace(go.Scatter3d(
        x=[0, 0], y=[0, 0], z=[0, volume_shape[2]//2],
        mode='lines', line=dict(color='black', width=2), name='Z (Vertical)'
    ))
    fig_3d.add_trace(go.Scatter3d(
        x=[volume_shape[0]//2, volume_shape[0]//2], y=[0, 0], z=[0, 10],
        mode='lines+text', text=['θ=0°, φ=0°'], textposition="top right",
        line=dict(color='gray', width=2), name='Initial Angio Position'
    ))

    fig_3d.update_layout(
        title=f"3D Reconstruction: Vessel and Shadows (Loss: {total_loss:.4f})",
        scene=dict(
            xaxis=dict(range=[0, volume_shape[0]], title='X (Left-Right)'),
            yaxis=dict(range=[0, volume_shape[1]], title='Y (Head-Feet)'),
            zaxis=dict(range=[0, volume_shape[2]], title='Z (Vertical)')
        ),
        width=800, height=800
    )
    fig_3d.show()

    # 2D-проекции
    fig2, axs = plt.subplots(2, 2, figsize=(10, 10))

    # Проекция 1: сосуд и shadow1
    axs[0, 0].imshow(proj_T1.detach().numpy(), cmap='Reds', alpha=0.7)
    axs[0, 0].imshow(proj1.detach().numpy(), cmap='Blues', alpha=0.5)
    axs[0, 0].set_title(f'Proj_T1 vs Proj1 (Loss: {loss1:.4f})')

    # Разница между Proj_T1 и Proj1
    diff1 = torch.abs(proj_T1 - proj1)
    axs[0, 1].imshow(diff1.detach().numpy(), cmap='hot')
    axs[0, 1].set_title('Difference Proj_T1 - Proj1')

    # Проекция 2: сосуд и shadow2
    axs[1, 0].imshow(proj_T2.detach().numpy(), cmap='Reds', alpha=0.7)
    axs[1, 0].imshow(proj2.detach().numpy(), cmap='Blues', alpha=0.5)
    axs[1, 0].set_title(f'Proj_T2 vs Proj2 (Loss: {loss2:.4f})')

    # Разница между Proj_T2 и Proj2
    diff2 = torch.abs(proj_T2 - proj2)
    axs[1, 1].imshow(diff2.detach().numpy(), cmap='hot')
    axs[1, 1].set_title('Difference Proj_T2 - Proj2')

    plt.suptitle(f'Total Loss: {total_loss:.4f} (phi_T={phi_T_adjusted:.1f}, theta_T={theta_T_adjusted:.1f})')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# Запуск интерактивной визуализации с подсказками
interact(update_visualization,
         phi_T=(-180.0, 180.0, 5.0),
         theta_T=(-180.0, 180.0, 5.0),
         shift_x=(-50, 50, 1),
         shift_y=(-50, 50, 1),
         z_offset=(-50, 50, 1),
         invert_phi=False,
         invert_theta=False)

In [ ]:
def compute_total_error(phi_T, theta_T, T, proj1, proj2, phi1, theta1, phi2, theta2, volume_shape):
    """Вычисляет общую ошибку между проекциями T и эталонными проекциями."""
    T_rotated = rotate_tensor(T, phi_T, theta_T, volume_shape)
    proj_T1 = project_3d_model(T_rotated, phi1, theta1, volume_shape)
    proj_T2 = project_3d_model(T_rotated, phi2, theta2, volume_shape)

    # Нормализация размеров
    if proj_T1.shape != proj1.shape:
        proj_T1 = torch.nn.functional.interpolate(
            proj_T1.unsqueeze(0).unsqueeze(0),
            size=proj1.shape,
            mode='bilinear',
            align_corners=True
        ).squeeze()
    if proj_T2.shape != proj2.shape:
        proj_T2 = torch.nn.functional.interpolate(
            proj_T2.unsqueeze(0).unsqueeze(0),
            size=proj2.shape,
            mode='bilinear',
            align_corners=True
        ).squeeze()

    error1 = torch.mean((proj_T1 - proj1) ** 2)
    error2 = torch.mean((proj_T2 - proj2) ** 2)
    return error1 + error2

# Эталонные проекции
proj1 = torch.max(shadow1, dim=2)[0]  # Синяя проекция
proj2 = torch.max(shadow2, dim=2)[0]  # Зеленая проекция

# Средние углы ангиографа
phi_avg = (-100.0 + (-99.30917358398438)) / 2  # ~ -99.65
theta_avg = (-130.0 + (-16.34215545654297)) / 2  # ~ -73.17

volume_shape = (128, 128, 128)
best_phi_T, best_theta_T = phi_avg, theta_avg
min_error = float('inf')

# Используем выровненный T
T = T_aligned

for phi_T in np.arange(phi_avg - 20, phi_avg + 21, 5):
    for theta_T in np.arange(theta_avg - 20, theta_avg + 21, 5):
        error = compute_total_error(phi_T, theta_T, T, proj1, proj2, phi1, theta1, phi2, theta2, volume_shape)
        if error < min_error:
            min_error = error
            best_phi_T, best_theta_T = phi_T, theta_T

print(f"Лучшие углы для T: phi_T={best_phi_T}, theta_T={best_theta_T}")

In [ ]:
from kemsekov_torch.residual import ResidualBlock
correct = T.unsqueeze(0).unsqueeze(0)

# Определение модели
model = torch.nn.Sequential(
    ResidualBlock(in_channels=2, out_channels=8, stride=2, dimensions=3),
    ResidualBlock(in_channels=8, out_channels=16, stride=2, dimensions=3),
    ResidualBlock(in_channels=16, out_channels=32, stride=2, dilation=[1]*8+[2]*8+[4]*16, dimensions=3),
    *[ResidualBlock(in_channels=32, out_channels=[8, 32], dimensions=3) for _ in range(5)],
    ResidualBlock(in_channels=32, out_channels=16, stride=2, dimensions=3).transpose(),
    ResidualBlock(in_channels=16, out_channels=8, stride=2, dimensions=3).transpose(),
    ResidualBlock(in_channels=8, out_channels=1, stride=2, dimensions=3).transpose(),
)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
volume_2ch = volume_2ch.to(device)
correct = correct.to(device)

# Обучение
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()
num_epochs = 100

model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    output = model(volume_2ch)
    loss = criterion(output, correct)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.6f}")

# Тестирование и визуализация
model.eval()
with torch.no_grad():
    predicted = model(volume_2ch).squeeze(0).squeeze(0)  # (128, 128, 128)
    print("Shape of predicted:", predicted.shape)
    print("Max value in predicted:", predicted.max().item())
    print("Min value in predicted:", predicted.min().item())
    nonzero_indices = (predicted > 0.01).nonzero(as_tuple=False)  # Порог для визуализации
    print("Number of nonzero points in predicted (threshold > 0.01):", len(nonzero_indices))
    if len(nonzero_indices) > 0:
        values = predicted[nonzero_indices[:, 0], nonzero_indices[:, 1], nonzero_indices[:, 2]]
        x_coords = nonzero_indices[:, 0].tolist()
        y_coords = nonzero_indices[:, 1].tolist()
        z_coords = nonzero_indices[:, 2].tolist()
        radii = values.tolist()
        fig = go.Figure(data=[go.Scatter3d(
            x=x_coords, y=y_coords, z=z_coords, mode='markers',
            marker=dict(size=3, color=radii, colorscale='Viridis', opacity=0.8, colorbar=dict(title="Predicted Value"))
        )])
        fig.update_layout(
            title="Predicted 3D Reconstruction",
            scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode='cube'),
            width=700, height=700,
            template="plotly_dark"
        )
        fig.show()
    else:
        print("No significant nonzero values (> 0.01) in predicted tensor to visualize!")

# Визуализация оригинального изображения (T)
nonzero_indices_orig = T.nonzero(as_tuple=False)
values_orig = T[nonzero_indices_orig[:, 0], nonzero_indices_orig[:, 1], nonzero_indices_orig[:, 2]]
x_coords_orig, y_coords_orig, z_coords_orig, radii_orig = [i.tolist() for i in [nonzero_indices_orig[:, 0], nonzero_indices_orig[:, 1], nonzero_indices_orig[:, 2], values_orig]]

fig_orig = go.Figure(data=[go.Scatter3d(
    x=x_coords_orig, y=y_coords_orig, z=z_coords_orig, mode='markers',
    marker=dict(size=[r * 100 for r in radii_orig], color=radii_orig, colorscale='Viridis', opacity=0.8, colorbar=dict(title="Original Radius"))
)])
fig_orig.update_layout(
    title="Original 3D Visualization of Tensor T",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode='cube'),
    width=700, height=700,
    template="plotly_dark"
)
fig_orig.show()

In [ ]:
# Визуализация оригинального изображения (T)
nonzero_indices_orig = T.nonzero(as_tuple=False)
values_orig = T[nonzero_indices_orig[:, 0], nonzero_indices_orig[:, 1], nonzero_indices_orig[:, 2]]
x_coords_orig, y_coords_orig, z_coords_orig, radii_orig = [i.tolist() for i in [nonzero_indices_orig[:, 0], nonzero_indices_orig[:, 1], nonzero_indices_orig[:, 2], values_orig]]

fig_orig = go.Figure(data=[go.Scatter3d(
    x=x_coords_orig, y=y_coords_orig, z=z_coords_orig, mode='markers',
    marker=dict(size=[r * 100 for r in radii_orig], color=radii_orig, colorscale='Viridis', opacity=0.8, colorbar=dict(title="Original Radius"))
)])
fig_orig.update_layout(
    title="Original 3D Visualization of Tensor T",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode='cube'),
    width=700, height=700,
    template="plotly_dark"
)
fig_orig.show()

In [ ]:
volume_2ch = volume_2ch.unsqueeze(0)
volume_2ch.shape

In [ ]:
with torch.no_grad():
    predicted = model(volume_2ch).squeeze(0).squeeze(0)
    print("Max value in predicted:", predicted.max().item())
    print("Min value in predicted:", predicted.min().item())

Пример того чего хотим достичь

In [ ]:
from kemsekov_torch.residual import ResidualBlock

def minmax(x):
    x-=x.min()
    x/=x.max()

V = torch.randn((4,300,4))
minmax(V[...,0])
minmax(V[...,1])
minmax(V[...,2])
minmax(V[...,3])
# V


model = torch.nn.Sequential(
    ResidualBlock(
        in_channels=2,
        out_channels=8,
        stride=2,
        dimensions=3
    ),
    ResidualBlock(
        in_channels=8,
        out_channels=16,
        stride=2,
        dimensions=3
    ),
    ResidualBlock(
        in_channels=16,
        out_channels=32,
        stride=2,
        dilation=[1]*8+[2]*8+[4]*16,
        dimensions=3
    ),
    *[ResidualBlock(
        in_channels=32,
        out_channels=[8,32],
        dimensions=3
    ) for _ in range(5)],
    ResidualBlock(
        in_channels=32,
        out_channels=16,
        stride=2,
        dimensions=3
    ).transpose(),
    ResidualBlock(
        in_channels=16,
        out_channels=8,
        stride=2,
        dimensions=3
    ).transpose(),
    ResidualBlock(
        in_channels=8,
        out_channels=1,
        stride=2,
        dimensions=3
    ).transpose(),
)

T = torch.randn((1,128,128,128))

x = 0.3
y=0.7
z=0.1
R=3

pos = int(x*128),int(y*128),int(z*128)
pos = torch.tensor([0,1,2,3,4]),torch.tensor([5,8,2,3,4]),torch.tensor([9,10,5,22,5])

for shiftx in [-1,0,1]:
    for shifty in [-1,0,1]:
        for shiftz in [-1,0,1]:
            position = [0,pos[0]+shiftx,pos[1]+shifty,pos[2]+shiftz]
            slice = T[*position]
            T[*position][slice==0]=R
            T[*position][slice!=0]+=R
            T[*position][slice!=0]/=2

# (batch_size, channels, x,y,z)
input = torch.randn((1,2,128,128,128))
correct = torch.randn((1,1,128,128,128))

torch.nn.functional.mse_loss(model(input),correct)

In [ ]:
# Добавляем batch_size и channels
T = T.unsqueeze(0).unsqueeze(0)  # (1, 1, 128, 128, 128)

# Пример входных данных для модели
input = torch.randn((1, 2, 128, 128, 128))  # (batch_size, channels, x, y, z)
correct = T  # (1, 1, 128, 128, 128)

# Пример вызова функции потерь
loss = torch.nn.functional.mse_loss(model(input), correct)